In [1]:
import os
from pathlib import Path
import pandas as pd
import requests

# Look for .env in the parent directory if your notebook is in a subfolder (e.g., notebooks/)
env_path = Path("../../.env") if Path("../../.env").exists() else Path(".env")
for line in env_path.read_text().splitlines() if env_path.exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
API_KEY = os.environ.get("RAWTREE_API_KEY")
TEAM = os.environ.get("TEAM_ID")

if not API_KEY:
    raise ValueError("RAWTREE_API_KEY is missing! Check your .env file location.")

http = requests.Session()
http.headers["Authorization"] = f"Bearer {API_KEY}"

In [2]:
def query(sql: str) -> pd.DataFrame:
    """Helper function to execute SQL queries against the RawTree API and return a DataFrame."""
    r = http.post(f"{API}/v1/query", json={"sql": sql}, timeout=600)
    r.raise_for_status()
    data = r.json().get("data", [])
    return pd.DataFrame(data)

print("Fetching all prediction data sources from RawTree...")

Fetching all prediction data sources from RawTree...


In [3]:
# All-time best sellers ranking (derived from historical purchases)
sales_sql = """
SELECT product_id, count() AS n
FROM shop.history_events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY n DESC
"""
best_df = query(sales_sql)
BEST_SELLERS: list[str] = best_df["product_id"].tolist() if not best_df.empty else []

# Current out-of-stock items (recommending these makes an answer invalid)
stock_sql = """
SELECT product_id::String AS product_id
FROM shop.catalog_updates
WHERE change::String IN ('out_of_stock', 'back_in_stock')
GROUP BY product_id
HAVING argMax(change::String, ts::DateTime64(3)) = 'out_of_stock'
"""
stock_df = query(stock_sql)
OUT_OF_STOCK: set[str] = set(stock_df["product_id"]) if not stock_df.empty else set()



In [4]:
# Co-views: For every product, the top 20 products most viewed in the same sessions
coviews_sql = """
SELECT a.product_id AS p, b.product_id AS q, count() AS n
FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS a
INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS b
  ON a.session_id = b.session_id
WHERE a.product_id != b.product_id
GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p
"""
coviews_df = query(coviews_sql)
COVIEWS_DICT: dict[str, list[str]] = (
    coviews_df.groupby("p").q.apply(list).to_dict() if not coviews_df.empty else {}
)


In [5]:

# Active incoming recommendation requests queue
requests_sql = """
SELECT request_id, ts, user_id, context, recent_views
FROM shop.requests
ORDER BY ts DESC
LIMIT 100
"""
RECENT_REQUESTS: pd.DataFrame = query(requests_sql)



In [6]:

# Historical session logs (contains impressions, clicks, propensity scores)[cite: 1]
history_sessions_sql = """
SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
FROM shop.history_sessions
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_SESSIONS: pd.DataFrame = query(history_sessions_sql)

# Granular historical event logs (individual views and purchases for matrix factorization / item similarity)
history_events_sql = """
SELECT session_id, user_id, event_type, product_id, ts
FROM shop.history_events
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_EVENTS: pd.DataFrame = query(history_events_sql)


In [7]:
catalog = query("SELECT * FROM shop.catalog")
feedback_df = query(f"SELECT * FROM {TEAM}.feedback ORDER BY ts DESC LIMIT 1000")

In [8]:
# Summary of loaded variables
if __name__ == "__main__":
    print("\n--- Data Loaded Successfully ---")
    print(f"• BEST_SELLERS: {len(BEST_SELLERS)} products loaded")
    print(f"• OUT_OF_STOCK: {len(OUT_OF_STOCK)} items currently unavailable")
    print(f"• COVIEWS_DICT: {len(COVIEWS_DICT)} products have co-view maps")
    print(f"• RECENT_REQUESTS: {len(RECENT_REQUESTS)} live request rows fetched")
    print(f"• HISTORY_SESSIONS: {len(HISTORY_SESSIONS)} session rows fetched")
    print(f"• HISTORY_EVENTS: {len(HISTORY_EVENTS)} event rows fetched")


--- Data Loaded Successfully ---
• BEST_SELLERS: 4809 products loaded
• OUT_OF_STOCK: 36 items currently unavailable
• COVIEWS_DICT: 5000 products have co-view maps
• RECENT_REQUESTS: 100 live request rows fetched
• HISTORY_SESSIONS: 10000 session rows fetched
• HISTORY_EVENTS: 10000 event rows fetched


---

In [9]:
bestSellers = pd.DataFrame(BEST_SELLERS)
outOfStock = pd.DataFrame(OUT_OF_STOCK)
coviewsDict = COVIEWS_DICT
recentRequests = pd.DataFrame(RECENT_REQUESTS)
historySessions = pd.DataFrame(HISTORY_SESSIONS)
historyEvents = pd.DataFrame(HISTORY_EVENTS)
catalog = catalog
feedback = feedback_df


In [10]:
positive_interactions_df = historySessions[
    historySessions['clicked'].notna() | historySessions['purchased'].notna()
]

positive_interactions_df.head()

,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
13,r_41387df0fa342ac3,2026-11-14 06:59:12.574000000,u_64815156,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_767d56', 'ts': '2026-11-14 ...","[p_0210c5, p_fe5189, p_170f13, p_b49b85, p_659...","[0.264415, 0.00542, 0.040626, 0.059437, 0.0195...",p_0210c5,p_0210c5
53,r_8616db58b3e8a8e9,2026-11-14 06:57:20.528000000,u_b117768f,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_789636', 'ts': '2026-11-14 ...","[p_aa3b8d, p_b69360, p_19bf96, p_47f703, p_ca8...","[0.068108, 0.027105, 0.003496, 0.001432, 0.003...",p_b69360,NaN
56,r_9d2bdcef4ce3db64,2026-11-14 06:57:14.386000000,u_34b5030a,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_2d900c', 'ts': '2026-11-14 ...","[p_0210c5, p_1036d3, p_55ab57, p_42fcad, p_0c3...","[0.264415, 0.236282, 0.139664, 0.024399, 0.058...",p_659551,NaN
93,r_7c89544a7c2c68e1,2026-11-14 06:54:48.939000000,u_82d8b488,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_dc2f02', 'ts': '2026-11-14 ...","[p_1e3f05, p_d94e20, p_2decec, p_3e2959, p_93a...","[0.036852, 0.057612, 0.004867, 0.006174, 0.012...",p_1e3f05,NaN
99,r_4d606bdcc2924ac2,2026-11-14 06:54:37.011000000,u_f6e1e092,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_abc61d', 'ts': '2026-11-14 ...","[p_ac4b0d, p_8ca8ed, p_f3bf99, p_692114, p_2c8...","[0.056724, 0.232049, 0.005473, 0.021444, 0.314...",p_ac4b0d,NaN


In [11]:
from featureConstructor import FeatureConstructor

Loading saved model factors...


/home/camoraz/Code/team18/team-18/.venv/lib/python3.13/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 22 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [12]:
import importlib
import featureConstructor
importlib.reload(featureConstructor)
from featureConstructor import FeatureConstructor

# Disable userItemFeatures (or any other block) at initialization
fc = FeatureConstructor(
    query_function=query,
    catalog=catalog,
    coview_dict=coviewsDict,
    best_sellers=bestSellers,
    out_of_stock=outOfStock,
    use_user_item_features=False,
)

X, y_click, y_buy, sample_weights = fc.generate_xy(
    interactions_df=positive_interactions_df,
    history_sessions=historySessions,
    max_examples=2000,
    random_state=42,  # Omit random_state to just take the first 2,000 in order
)

TypeError: FeatureConstructor.generate_xy() got an unexpected keyword argument 'max_examples'

In [ ]:
test_payload = {
    "product_id": "p_001897",
    "brand": "Vedo",
    "category": "Sports & Outdoors",
    "created_at": "2026-01-21 07:00:00.000000000",
    "description": "Vedo Yoga Mats with organic ingredients and ex...",
    "is_active": True,
    "price_eur": 14.99,
    "subcategory": "Yoga Mats",
    "ts": "2026-11-14 14:58:05.378000000",
    "user_id": "u_f0b1f882",
    "is_new_user": False,
    "context": {
        "device": "ios", 
        "page": "product", 
        "region": "US",
    },
    "recent_views": [],
    "coview_dict": {},
    "query_function": query,
    "catalog": catalog
}

X = featureConstructor.constructFeatures(test_payload)

NameError: name 'featureConstructor' is not defined

In [ ]:
fc = FeatureConstructor(
    query_function=query,
    catalog=catalog,
    coview_dict=coviewsDict,
    best_sellers=bestSellers,
    out_of_stock=outOfStock,
)

X, y_click, y_buy, sample_weights = fc.generate_xy(
    interactions_df=positive_interactions_df,
    history_sessions=historySessions,
)

model_click.fit(X, y_click, sample_weight=sample_weights)
model_buy.fit(X, y_buy, sample_weight=sample_weights)

83
